# 25_E17 - Bridge Spring Boot hacia servicio IA

Este notebook genera un scaffold de integración Spring Boot → servicio Python FastAPI, usando el contrato validado en E16.

No entrena modelos y no ejecuta inferencia. Su objetivo es pasar de la validación de servicio a una estructura inicial de backend consumible por frontend.

Objetivos:

- Leer el reporte E16 y el contrato `/agent/report`.
- Generar archivos Java/Spring Boot para cliente, controller, service y DTOs.
- Generar documentación de integración.
- Validar por smoke test estático que el scaffold tenga los archivos esperados.
- Dejar listo un prompt para Codex/backend.


In [1]:
from pathlib import Path
import json
import os
import textwrap
import pandas as pd

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"

E16_ROOT = PFI_ROOT / "results" / "E16_backend_integration_smoke"
E17_ROOT = PFI_ROOT / "results" / "E17_spring_boot_bridge_scaffold"

DOCS_ROOT = PFI_ROOT / "docs"
BACKLOG_ROOT = REPO_ROOT / "backlogProducto"
BRIDGE_ROOT = REPO_ROOT / "backend_bridge_spring"

for p in [E17_ROOT, DOCS_ROOT, BACKLOG_ROOT, BRIDGE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

E16_REPORT_PATH = E16_ROOT / "E16_backend_integration_report.json"
E16_SAMPLE_PATH = E16_ROOT / "E16_agent_report_response_sample.json"
E16_CONTRACT_PATH = E16_ROOT / "E16_backend_contract_summary.json"

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("E16_REPORT_PATH:", E16_REPORT_PATH, "| exists:", E16_REPORT_PATH.exists())
print("E16_SAMPLE_PATH:", E16_SAMPLE_PATH, "| exists:", E16_SAMPLE_PATH.exists())
print("BRIDGE_ROOT:", BRIDGE_ROOT)
assert REPO_ROOT.exists(), "No existe el repo en Drive"
assert E16_REPORT_PATH.exists(), "Falta el reporte E16. Corré E16 primero."


Mounted at /content/drive
PFI_ROOT: /content/drive/MyDrive/PFI_MVP
REPO_ROOT: /content/drive/MyDrive/PFI_MVP/repo | exists: True
E16_REPORT_PATH: /content/drive/MyDrive/PFI_MVP/results/E16_backend_integration_smoke/E16_backend_integration_report.json | exists: True
E16_SAMPLE_PATH: /content/drive/MyDrive/PFI_MVP/results/E16_backend_integration_smoke/E16_agent_report_response_sample.json | exists: True
BRIDGE_ROOT: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring


In [2]:
# Cargar reporte y muestra E16

e16_report = json.loads(E16_REPORT_PATH.read_text(encoding="utf-8"))

if E16_SAMPLE_PATH.exists():
    e16_sample = json.loads(E16_SAMPLE_PATH.read_text(encoding="utf-8"))
else:
    # Fallback usando el reporte completo
    e16_sample = {
        "summary": e16_report.get("summary", {}),
        "items": [],
        "markdown": "",
    }

print("E16 decision:", e16_report.get("decision"))
print("E16 checks:", json.dumps(e16_report.get("checks", {}), indent=2, ensure_ascii=False))
print("Sample keys:", list(e16_sample.keys()))
print("Summary:", json.dumps(e16_sample.get("summary", {}), indent=2, ensure_ascii=False))
print("Items:", len(e16_sample.get("items", [])))


E16 decision: backend_integration_smoke_ready_for_spring_boot_bridge
E16 checks: {
  "package_import_ok": true,
  "fastapi_app_import_ok": true,
  "all_endpoints_ok": true,
  "contract_checks_passed": true,
  "human_review_required": true
}
Sample keys: ['summary', 'markdown', 'items']
Summary: {
  "total_items": 12,
  "plane_distribution": {
    "axial": 6,
    "sagittal": 6
  },
  "priority_distribution": {
    "baja": 6,
    "media": 5,
    "alta": 1
  },
  "status_distribution": {
    "listo_para_revision_estandar": 6,
    "requiere_revision_con_atencion": 5,
    "requiere_revision_prioritaria": 1
  },
  "mean_fg_confidence": 0.8962119023005167,
  "mean_dice_macro_useful_classes": 0.8659508906844443
}
Items: 12


## 1. Generación del scaffold Spring Boot

El scaffold queda en:

```text
repo/backend_bridge_spring/
```

Está pensado para copiarse o integrarse dentro del backend Spring Boot real cuando definan la estructura final del backend.


In [3]:
# Contenido de archivos Java/Spring Boot

PACKAGE_BASE = "ar.edu.uade.pfi.ai"
PACKAGE_PATH = Path(*PACKAGE_BASE.split("."))

def write_file(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    print("Wrote:", path)

files = {}

files["README.md"] = """
# Backend Bridge Spring Boot - PFI AI Service

Este scaffold muestra cómo integrar un backend Spring Boot con el servicio Python FastAPI generado en E15/E16.

## Rol arquitectónico

- Python/FastAPI concentra IA, inferencia, quality flags y agente.
- Spring Boot actúa como backend de producto.
- El frontend consume Spring Boot, no directamente el servicio Python.

## Endpoints esperados del servicio Python

```text
GET /health
GET /models
GET /agent/worklist
GET /agent/report
```

## Endpoint propuesto en Spring Boot

```text
GET /api/ai/agent/report
```

## Configuración sugerida

```yaml
pfi:
  ai-service:
    base-url: http://localhost:8000
    timeout-ms: 10000
```

## Flujo

Frontend -> Spring Boot -> Python FastAPI -> Spring Boot -> Frontend

La revisión profesional sigue siendo obligatoria.
"""

files["application-ai-example.yml"] = """
pfi:
  ai-service:
    base-url: http://localhost:8000
    timeout-ms: 10000
"""

files["pom-webclient-snippet.xml"] = """
<!-- Dependencias sugeridas para el backend Spring Boot -->
<dependencies>
    <dependency>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-web</artifactId>
    </dependency>

    <dependency>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-webflux</artifactId>
    </dependency>

    <dependency>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-validation</artifactId>
    </dependency>
</dependencies>
"""

java_root = BRIDGE_ROOT / "src/main/java" / PACKAGE_PATH

files[str(Path("src/main/java") / PACKAGE_PATH / "config/AiServiceProperties.java")] = """
package ar.edu.uade.pfi.ai.config;

import org.springframework.boot.context.properties.ConfigurationProperties;

@ConfigurationProperties(prefix = "pfi.ai-service")
public class AiServiceProperties {

    private String baseUrl = "http://localhost:8000";
    private Integer timeoutMs = 10000;

    public String getBaseUrl() {
        return baseUrl;
    }

    public void setBaseUrl(String baseUrl) {
        this.baseUrl = baseUrl;
    }

    public Integer getTimeoutMs() {
        return timeoutMs;
    }

    public void setTimeoutMs(Integer timeoutMs) {
        this.timeoutMs = timeoutMs;
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "config/AiBridgeConfig.java")] = """
package ar.edu.uade.pfi.ai.config;

import java.time.Duration;

import org.springframework.boot.context.properties.EnableConfigurationProperties;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.web.reactive.function.client.WebClient;

@Configuration
@EnableConfigurationProperties(AiServiceProperties.class)
public class AiBridgeConfig {

    @Bean
    public WebClient aiServiceWebClient(AiServiceProperties properties) {
        return WebClient.builder()
                .baseUrl(properties.getBaseUrl())
                .build();
    }

    @Bean
    public Duration aiServiceTimeout(AiServiceProperties properties) {
        return Duration.ofMillis(properties.getTimeoutMs());
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "dto/AiAgentSummaryDto.java")] = """
package ar.edu.uade.pfi.ai.dto;

import java.util.Map;

public class AiAgentSummaryDto {

    private Integer totalItems;
    private Map<String, Integer> planeDistribution;
    private Map<String, Integer> priorityDistribution;
    private Map<String, Integer> statusDistribution;
    private Double meanFgConfidence;
    private Double meanDiceMacroUsefulClasses;

    public Integer getTotalItems() {
        return totalItems;
    }

    public void setTotalItems(Integer totalItems) {
        this.totalItems = totalItems;
    }

    public Map<String, Integer> getPlaneDistribution() {
        return planeDistribution;
    }

    public void setPlaneDistribution(Map<String, Integer> planeDistribution) {
        this.planeDistribution = planeDistribution;
    }

    public Map<String, Integer> getPriorityDistribution() {
        return priorityDistribution;
    }

    public void setPriorityDistribution(Map<String, Integer> priorityDistribution) {
        this.priorityDistribution = priorityDistribution;
    }

    public Map<String, Integer> getStatusDistribution() {
        return statusDistribution;
    }

    public void setStatusDistribution(Map<String, Integer> statusDistribution) {
        this.statusDistribution = statusDistribution;
    }

    public Double getMeanFgConfidence() {
        return meanFgConfidence;
    }

    public void setMeanFgConfidence(Double meanFgConfidence) {
        this.meanFgConfidence = meanFgConfidence;
    }

    public Double getMeanDiceMacroUsefulClasses() {
        return meanDiceMacroUsefulClasses;
    }

    public void setMeanDiceMacroUsefulClasses(Double meanDiceMacroUsefulClasses) {
        this.meanDiceMacroUsefulClasses = meanDiceMacroUsefulClasses;
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "dto/AiAgentItemDto.java")] = """
package ar.edu.uade.pfi.ai.dto;

import java.util.List;
import java.util.Map;

public class AiAgentItemDto {

    private String agentItemId;
    private String plane;
    private String modelKey;
    private String caseRef;
    private String figurePath;

    private Double foregroundRatio;
    private Integer nComponents;
    private List<Integer> presentClasses;
    private List<String> flags;
    private Double meanConfidence;
    private Double meanFgConfidence;

    private String agentStatus;
    private String reviewPriority;
    private List<String> agentReasons;
    private String recommendedAction;
    private Boolean humanReviewRequired;

    private Map<String, Object> extra;

    public String getAgentItemId() {
        return agentItemId;
    }

    public void setAgentItemId(String agentItemId) {
        this.agentItemId = agentItemId;
    }

    public String getPlane() {
        return plane;
    }

    public void setPlane(String plane) {
        this.plane = plane;
    }

    public String getModelKey() {
        return modelKey;
    }

    public void setModelKey(String modelKey) {
        this.modelKey = modelKey;
    }

    public String getCaseRef() {
        return caseRef;
    }

    public void setCaseRef(String caseRef) {
        this.caseRef = caseRef;
    }

    public String getFigurePath() {
        return figurePath;
    }

    public void setFigurePath(String figurePath) {
        this.figurePath = figurePath;
    }

    public Double getForegroundRatio() {
        return foregroundRatio;
    }

    public void setForegroundRatio(Double foregroundRatio) {
        this.foregroundRatio = foregroundRatio;
    }

    public Integer getNComponents() {
        return nComponents;
    }

    public void setNComponents(Integer nComponents) {
        this.nComponents = nComponents;
    }

    public List<Integer> getPresentClasses() {
        return presentClasses;
    }

    public void setPresentClasses(List<Integer> presentClasses) {
        this.presentClasses = presentClasses;
    }

    public List<String> getFlags() {
        return flags;
    }

    public void setFlags(List<String> flags) {
        this.flags = flags;
    }

    public Double getMeanConfidence() {
        return meanConfidence;
    }

    public void setMeanConfidence(Double meanConfidence) {
        this.meanConfidence = meanConfidence;
    }

    public Double getMeanFgConfidence() {
        return meanFgConfidence;
    }

    public void setMeanFgConfidence(Double meanFgConfidence) {
        this.meanFgConfidence = meanFgConfidence;
    }

    public String getAgentStatus() {
        return agentStatus;
    }

    public void setAgentStatus(String agentStatus) {
        this.agentStatus = agentStatus;
    }

    public String getReviewPriority() {
        return reviewPriority;
    }

    public void setReviewPriority(String reviewPriority) {
        this.reviewPriority = reviewPriority;
    }

    public List<String> getAgentReasons() {
        return agentReasons;
    }

    public void setAgentReasons(List<String> agentReasons) {
        this.agentReasons = agentReasons;
    }

    public String getRecommendedAction() {
        return recommendedAction;
    }

    public void setRecommendedAction(String recommendedAction) {
        this.recommendedAction = recommendedAction;
    }

    public Boolean getHumanReviewRequired() {
        return humanReviewRequired;
    }

    public void setHumanReviewRequired(Boolean humanReviewRequired) {
        this.humanReviewRequired = humanReviewRequired;
    }

    public Map<String, Object> getExtra() {
        return extra;
    }

    public void setExtra(Map<String, Object> extra) {
        this.extra = extra;
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "dto/AiAgentReportResponseDto.java")] = """
package ar.edu.uade.pfi.ai.dto;

import java.util.List;

public class AiAgentReportResponseDto {

    private AiAgentSummaryDto summary;
    private String markdown;
    private List<AiAgentItemDto> items;

    public AiAgentSummaryDto getSummary() {
        return summary;
    }

    public void setSummary(AiAgentSummaryDto summary) {
        this.summary = summary;
    }

    public String getMarkdown() {
        return markdown;
    }

    public void setMarkdown(String markdown) {
        this.markdown = markdown;
    }

    public List<AiAgentItemDto> getItems() {
        return items;
    }

    public void setItems(List<AiAgentItemDto> items) {
        this.items = items;
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "client/AiServiceClient.java")] = """
package ar.edu.uade.pfi.ai.client;

import java.time.Duration;

import org.springframework.stereotype.Component;
import org.springframework.web.reactive.function.client.WebClient;

import ar.edu.uade.pfi.ai.dto.AiAgentReportResponseDto;

@Component
public class AiServiceClient {

    private final WebClient webClient;
    private final Duration timeout;

    public AiServiceClient(WebClient aiServiceWebClient, Duration aiServiceTimeout) {
        this.webClient = aiServiceWebClient;
        this.timeout = aiServiceTimeout;
    }

    public AiAgentReportResponseDto getAgentReport() {
        return webClient.get()
                .uri("/agent/report")
                .retrieve()
                .bodyToMono(AiAgentReportResponseDto.class)
                .block(timeout);
    }

    public String health() {
        return webClient.get()
                .uri("/health")
                .retrieve()
                .bodyToMono(String.class)
                .block(timeout);
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "service/AiAgentBridgeService.java")] = """
package ar.edu.uade.pfi.ai.service;

import org.springframework.stereotype.Service;

import ar.edu.uade.pfi.ai.client.AiServiceClient;
import ar.edu.uade.pfi.ai.dto.AiAgentReportResponseDto;

@Service
public class AiAgentBridgeService {

    private final AiServiceClient aiServiceClient;

    public AiAgentBridgeService(AiServiceClient aiServiceClient) {
        this.aiServiceClient = aiServiceClient;
    }

    public AiAgentReportResponseDto getAgentReport() {
        return aiServiceClient.getAgentReport();
    }

    public String health() {
        return aiServiceClient.health();
    }
}
"""

files[str(Path("src/main/java") / PACKAGE_PATH / "controller/AiAgentController.java")] = """
package ar.edu.uade.pfi.ai.controller;

import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.RequestMapping;
import org.springframework.web.bind.annotation.RestController;

import ar.edu.uade.pfi.ai.dto.AiAgentReportResponseDto;
import ar.edu.uade.pfi.ai.service.AiAgentBridgeService;

@RestController
@RequestMapping("/api/ai")
public class AiAgentController {

    private final AiAgentBridgeService aiAgentBridgeService;

    public AiAgentController(AiAgentBridgeService aiAgentBridgeService) {
        this.aiAgentBridgeService = aiAgentBridgeService;
    }

    @GetMapping("/health")
    public ResponseEntity<String> health() {
        return ResponseEntity.ok(aiAgentBridgeService.health());
    }

    @GetMapping("/agent/report")
    public ResponseEntity<AiAgentReportResponseDto> getAgentReport() {
        return ResponseEntity.ok(aiAgentBridgeService.getAgentReport());
    }
}
"""


In [4]:
# Escribir scaffold
for rel_path, content in files.items():
    write_file(BRIDGE_ROOT / rel_path, content)

print("Archivos generados:", len(files))


Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/README.md
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/application-ai-example.yml
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/pom-webclient-snippet.xml
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/config/AiServiceProperties.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/config/AiBridgeConfig.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentSummaryDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentItemDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentReportResponseDto.java
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/client/AiS

## 2. Documentación E17

Se generan documentos de apoyo para backend, frontend y Codex.


In [5]:
# Documentación y contrato

frontend_contract_md = f"""
# E17 - Contrato frontend para reporte IA

Endpoint Spring Boot propuesto:

```text
GET /api/ai/agent/report
```

Respuesta esperada:

```json
{{
  "summary": {{
    "totalItems": 12,
    "planeDistribution": {{"axial": 6, "sagittal": 6}},
    "priorityDistribution": {{"baja": 6, "media": 5, "alta": 1}},
    "statusDistribution": {{"listo_para_revision_estandar": 6}}
  }},
  "markdown": "...",
  "items": [
    {{
      "agentItemId": "AXIAL_001",
      "plane": "axial",
      "modelKey": "axial_t2_alkafri",
      "caseRef": "case=7 | D=3",
      "figurePath": "...",
      "foregroundRatio": 0.067,
      "nComponents": 5,
      "presentClasses": [1,2,3,4,5],
      "flags": [],
      "meanConfidence": 0.989,
      "meanFgConfidence": 0.954,
      "agentStatus": "listo_para_revision_estandar",
      "reviewPriority": "baja",
      "agentReasons": [],
      "recommendedAction": "Enviar a revisión profesional estándar",
      "humanReviewRequired": true
    }}
  ]
}}
```

Uso sugerido en UI:

- Tabla worklist por prioridad.
- Filtro por plano.
- Badge de estado.
- Link o preview de overlay.
- Panel lateral con razones del agente.
- Botón de validación profesional.
"""

spring_contract_md = """
# E17 - Bridge Spring Boot hacia servicio Python

## Servicio Python

Debe estar levantado en:

```text
http://localhost:8000
```

Endpoints usados:

```text
GET /health
GET /agent/report
```

## Spring Boot

Endpoint propuesto para frontend:

```text
GET /api/ai/agent/report
```

## Responsabilidades

Spring Boot:

- Maneja autenticación/autorización futura.
- Expone contrato estable al frontend.
- Invoca el servicio Python.
- Maneja errores de conectividad.
- Persiste resultados si se agrega base de datos.

Python FastAPI:

- Ejecuta lógica IA/agente.
- Devuelve worklist, priorities y reportes.
- Mantiene human_review_required = true.
"""

codex_prompt_md = """
# Prompt Codex - E17 Spring Boot Bridge

Trabajá sobre el repo del proyecto PFI. Objetivo: integrar el scaffold `backend_bridge_spring/` dentro del backend Spring Boot real.

Contexto:
- Existe un servicio Python FastAPI en `ai_service/`.
- E16 validó `/health`, `/models`, `/agent/worklist` y `/agent/report`.
- El endpoint principal a consumir es `GET /agent/report`.
- La revisión humana es obligatoria.

Tareas:
1. Detectar la estructura real del backend Spring Boot.
2. Copiar/adaptar los DTOs de `backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto`.
3. Agregar `AiServiceProperties`, `AiBridgeConfig`, `AiServiceClient`, `AiAgentBridgeService` y `AiAgentController`.
4. Configurar `pfi.ai-service.base-url`.
5. Exponer `GET /api/ai/agent/report`.
6. Manejar errores si el servicio Python no responde.
7. Agregar tests mínimos de controller/service si la estructura lo permite.

No cambiar la lógica IA. No mover modelos. Solo crear el bridge backend.
"""

conclusion_md = f"""
# E17 - Cierre de scaffold Spring Boot bridge

## Objetivo

Generar una primera estructura Java/Spring Boot para consumir el servicio Python validado en E16.

## Resultado

Se generó el scaffold en:

```text
{BRIDGE_ROOT}
```

## Decisión arquitectónica

El backend Java no ejecuta modelos de IA. El backend consume el servicio Python por HTTP y expone al frontend un endpoint estable.

## Endpoint propuesto

```text
GET /api/ai/agent/report
```

## Próximo paso

Integrar este scaffold en el backend Spring Boot real y validar el flujo:

```text
Frontend -> Spring Boot -> Python FastAPI -> Spring Boot -> Frontend
```
"""

write_file(DOCS_ROOT / "E17_frontend_payload_contract.md", frontend_contract_md)
write_file(DOCS_ROOT / "E17_spring_boot_bridge_contract.md", spring_contract_md)
write_file(DOCS_ROOT / "E17_codex_spring_bridge_prompt.md", codex_prompt_md)
write_file(DOCS_ROOT / "E17_spring_boot_bridge_conclusion.md", conclusion_md)

# Backlog local
write_file(BACKLOG_ROOT / "E17_plan.md", """
# E17 - Bridge Spring Boot hacia servicio IA

Estado: en progreso/hecho por scaffold.

Objetivo: generar un scaffold Java/Spring Boot para consumir el servicio Python FastAPI validado en E16.

Salidas:
- backend_bridge_spring/
- docs/E17_spring_boot_bridge_conclusion.md
- docs/E17_codex_spring_bridge_prompt.md
- results/E17_spring_boot_bridge_scaffold/

Decision: Spring Boot consume la IA por HTTP. Python conserva inferencia/agente.
""")


Wrote: /content/drive/MyDrive/PFI_MVP/docs/E17_frontend_payload_contract.md
Wrote: /content/drive/MyDrive/PFI_MVP/docs/E17_spring_boot_bridge_contract.md
Wrote: /content/drive/MyDrive/PFI_MVP/docs/E17_codex_spring_bridge_prompt.md
Wrote: /content/drive/MyDrive/PFI_MVP/docs/E17_spring_boot_bridge_conclusion.md
Wrote: /content/drive/MyDrive/PFI_MVP/repo/backlogProducto/E17_plan.md


## 3. Smoke test estático del scaffold

Este smoke test no compila Java. Valida que el scaffold exista, que los archivos esperados estén presentes y que el contrato mínimo se mantenga.


In [6]:
# Inventario de archivos generados

generated_files = sorted([p for p in BRIDGE_ROOT.rglob("*") if p.is_file()])
inventory_rows = []

for p in generated_files:
    inventory_rows.append({
        "relative_path": str(p.relative_to(REPO_ROOT)),
        "size_bytes": p.stat().st_size,
        "suffix": p.suffix,
    })

inventory_df = pd.DataFrame(inventory_rows)
inventory_path = E17_ROOT / "E17_bridge_file_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)

display(inventory_df)
print("Inventory:", inventory_path)


,relative_path,size_bytes,suffix
0,backend_bridge_spring/README.md,814,.md
1,backend_bridge_spring/application-ai-example.yml,77,.yml
2,backend_bridge_spring/pom-webclient-snippet.xml,536,.xml
3,backend_bridge_spring/src/main/java/ar/edu/uad...,991,.java
4,backend_bridge_spring/src/main/java/ar/edu/uad...,783,.java
5,backend_bridge_spring/src/main/java/ar/edu/uad...,606,.java
6,backend_bridge_spring/src/main/java/ar/edu/uad...,994,.java
7,backend_bridge_spring/src/main/java/ar/edu/uad...,3734,.java
8,backend_bridge_spring/src/main/java/ar/edu/uad...,696,.java
9,backend_bridge_spring/src/main/java/ar/edu/uad...,1732,.java


Inventory: /content/drive/MyDrive/PFI_MVP/results/E17_spring_boot_bridge_scaffold/E17_bridge_file_inventory.csv


In [7]:
# Validaciones de estructura

expected_files = [
    "backend_bridge_spring/README.md",
    "backend_bridge_spring/application-ai-example.yml",
    "backend_bridge_spring/pom-webclient-snippet.xml",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/config/AiServiceProperties.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/config/AiBridgeConfig.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentSummaryDto.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentItemDto.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/dto/AiAgentReportResponseDto.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/client/AiServiceClient.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/service/AiAgentBridgeService.java",
    "backend_bridge_spring/src/main/java/ar/edu/uade/pfi/ai/controller/AiAgentController.java",
]

checks = []
for rel in expected_files:
    p = REPO_ROOT / rel
    checks.append({
        "check": "file_exists",
        "target": rel,
        "ok": p.exists(),
        "size_bytes": p.stat().st_size if p.exists() else 0,
    })

# Contrato mínimo del sample E16
summary_keys = ["total_items", "plane_distribution", "priority_distribution", "status_distribution"]
item_keys = [
    "agent_item_id", "plane", "model_key", "case_ref", "figure_path",
    "foreground_ratio", "n_components", "present_classes", "flags",
    "mean_confidence", "mean_fg_confidence", "agent_status",
    "review_priority", "agent_reasons", "recommended_action", "human_review_required"
]

summary = e16_sample.get("summary", {})
items = e16_sample.get("items", [])
first_item = items[0] if items else {}

for key in summary_keys:
    checks.append({"check": "summary_key", "target": key, "ok": key in summary, "size_bytes": None})

for key in item_keys:
    checks.append({"check": "item_key", "target": key, "ok": key in first_item, "size_bytes": None})

checks_df = pd.DataFrame(checks)
checks_path = E17_ROOT / "E17_bridge_static_checks.csv"
checks_df.to_csv(checks_path, index=False)

display(checks_df)
print("Checks:", checks_path)
print("All OK:", bool(checks_df["ok"].all()))


,check,target,ok,size_bytes
0,file_exists,backend_bridge_spring/README.md,True,814.0
1,file_exists,backend_bridge_spring/application-ai-example.yml,True,77.0
2,file_exists,backend_bridge_spring/pom-webclient-snippet.xml,True,536.0
3,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,606.0
4,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,783.0
5,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,1732.0
6,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,3734.0
7,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,696.0
8,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,991.0
9,file_exists,backend_bridge_spring/src/main/java/ar/edu/uad...,True,601.0


Checks: /content/drive/MyDrive/PFI_MVP/results/E17_spring_boot_bridge_scaffold/E17_bridge_static_checks.csv
All OK: True


In [8]:
# Reporte final E17

report = {
    "notebook": "25_E17_spring_boot_bridge_scaffold",
    "goal": "generate Spring Boot bridge scaffold for Python AI service",
    "bridge_root": str(BRIDGE_ROOT),
    "source_e16_report": str(E16_REPORT_PATH),
    "source_e16_sample": str(E16_SAMPLE_PATH),
    "outputs": {
        "inventory_csv": str(inventory_path),
        "static_checks_csv": str(checks_path),
        "frontend_contract_md": str(DOCS_ROOT / "E17_frontend_payload_contract.md"),
        "spring_contract_md": str(DOCS_ROOT / "E17_spring_boot_bridge_contract.md"),
        "codex_prompt_md": str(DOCS_ROOT / "E17_codex_spring_bridge_prompt.md"),
        "conclusion_md": str(DOCS_ROOT / "E17_spring_boot_bridge_conclusion.md"),
    },
    "summary": {
        "generated_files": int(len(inventory_df)),
        "expected_files": int(len(expected_files)),
        "all_static_checks_ok": bool(checks_df["ok"].all()),
        "e16_total_items": summary.get("total_items"),
        "e16_priority_distribution": summary.get("priority_distribution"),
    },
    "architecture_decision": {
        "spring_boot_runs_product_backend": True,
        "python_service_runs_ai_agent": True,
        "frontend_consumes_spring_boot": True,
        "human_review_required": True,
    },
    "decision": "spring_boot_bridge_scaffold_ready_for_backend_integration",
}

report_path = E17_ROOT / "E17_bridge_scaffold_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print("Reporte E17:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))


Reporte E17: /content/drive/MyDrive/PFI_MVP/results/E17_spring_boot_bridge_scaffold/E17_bridge_scaffold_report.json
{
  "notebook": "25_E17_spring_boot_bridge_scaffold",
  "goal": "generate Spring Boot bridge scaffold for Python AI service",
  "bridge_root": "/content/drive/MyDrive/PFI_MVP/repo/backend_bridge_spring",
  "source_e16_report": "/content/drive/MyDrive/PFI_MVP/results/E16_backend_integration_smoke/E16_backend_integration_report.json",
  "source_e16_sample": "/content/drive/MyDrive/PFI_MVP/results/E16_backend_integration_smoke/E16_agent_report_response_sample.json",
  "outputs": {
    "inventory_csv": "/content/drive/MyDrive/PFI_MVP/results/E17_spring_boot_bridge_scaffold/E17_bridge_file_inventory.csv",
    "static_checks_csv": "/content/drive/MyDrive/PFI_MVP/results/E17_spring_boot_bridge_scaffold/E17_bridge_static_checks.csv",
    "frontend_contract_md": "/content/drive/MyDrive/PFI_MVP/docs/E17_frontend_payload_contract.md",
    "spring_contract_md": "/content/drive/MyDriv

## 4. Commit sugerido

Después de revisar los outputs, commitear desde Colab:

```bash
cd /content/drive/MyDrive/PFI_MVP/repo
git status
git add backend_bridge_spring/ backlogProducto/E17_plan.md notebooks/25_E17_spring_boot_bridge_scaffold.ipynb
git commit -m "Add E17 Spring Boot AI service bridge scaffold"
git push
```
